# 01 Piezometer locations

<!-- navigation -->
[<kbd>← 00 Set up project</kbd>](00%20Set%20up%20project.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>10 Import time series from Excel →</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb)

**Required** · minimal path: 00 → **01** → 10 → 20 → 51 → 52

Reads and checks the piezometer table, and writes the piezometers to the project geopackage, with a map per transect.

| | |
|---|---|
| **Input** | `piezometers.xlsx` (template), cross-section line in the transects metadata (for the map) |
| **Output** | `project_data.gpkg`, layer `piezometers` |

One row per piezometer filter with its transect, coordinates, filter screen, model layer and `x_crosssection`: the x-coordinate in the model, i.e. the distance from the boundary between foreland and hinterland (usually the inner toe of the levee), positive towards the hinterland. You determine this distance yourself, e.g. in a GIS.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

## 1. Read the piezometers

In [ ]:
piezometers = tlt.read_piezometer_table(project.general_file("piezometers"))
piezometers

## 2. x-coordinate and model layer
Check the distances and model layers per transect (the map below shows them as well).

In [ ]:
piezometers.sort_values(["transect", "x_crosssection", "modellayer"])[
    ["name", "transect", "x_crosssection", "modellayer", "screen_top", "screen_bottom", "use_calibration"]
]

## 3. Save to the project geopackage

In [ ]:
gdf = tlt.write_piezometers(
    piezometers,
    project.general_file("project_data"),
    layer=project.settings["paths"]["piezometer_layer"],
    crs=project.crs,
)
print(f"{len(gdf)} piezometers written to {project.relpath(project.general_file('project_data'))}")

## 4. Map per transect
Cross-section line (from the river side) and the piezometers with their `x_crosssection`.

In [ ]:
for transect, group in gdf.groupby("transect"):
    f, ax = tlt.plot_transect_map(group, line=project.crosssection(transect), crs=project.crs)  # source="brt": PDOK
    tlt.finalize_and_save(f, project.transect_dir(transect, "intermediate"), f"{project.name} - {transect}", tag="piezometers_map", file_label=transect)

---

<!-- navigation -->
[<kbd>← 00 Set up project</kbd>](00%20Set%20up%20project.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>10 Import time series from Excel →</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb)